In [0]:
%sql
LIST '/Volumes/workspace/default/s3_rag_and_roll/';

## Interacting with AWS s3

### Listing **External locations**

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

<span style="color:red">⚠️ Note: Different methods exist for connecting to S3 buckets, such as using Unity Catalog external locations or direct access via AWS credentials. Choose the method that best fits your security and compliance requirements.</span>

### Listing s3 bucket

#### Direct S3 Access bypassing Unity Catalog entirely

In [0]:
%fs
ls s3://rag-and-roll/

In [0]:
%sql
-- Drop the existing volume
DROP VOLUME IF EXISTS workspace.default.s3_rag_and_roll;

-- Recreate pointing to the S3 bucket root where files actually are
CREATE EXTERNAL VOLUME workspace.default.s3_rag_and_roll
LOCATION 's3://rag-and-roll/';

### Read sample file by direct access to the file

In [0]:
df = spark.read.csv('s3://rag-and-roll/fer2013.csv', header=True, inferSchema=True)
display(df.sample(.01))

### Read sample file by direct accessing the previously defined External Volume

In [0]:
%sql
-- Read CSV file
SELECT * FROM csv.`/Volumes/workspace/default/s3_rag_and_roll/fer2013.csv` LIMIT 10;

In [0]:
# Read with Spark
df = spark.read.csv('/Volumes/workspace/default/s3_rag_and_roll/fer2013.csv', header=True)
display(df.limit(10))

# Connecting to Postgresql DB exposed by AWS as Aurora Postgresql 

## Łączenie bezpośrednie po JDBC

In [0]:
aws_aurora_pg_endpoint = dbutils.secrets.get(scope="dvdrental", key="aws_aurora_pg_endpoint")
jdbc_url = f"jdbc:postgresql://{aws_aurora_pg_endpoint}:5432/dvdrental"
print(jdbc_url == "jdbc:postgresql://dvdrental-cluster-instance-1.c7mma2sss3r3.eu-west-1.rds.amazonaws.com:5432/dvdrental")
connection_properties = {
    "user": dbutils.secrets.get(scope="dvdrental", key="db_user"),
    "password": dbutils.secrets.get(scope="dvdrental", key="db_password"),
    "driver": "org.postgresql.Driver"
}

In [0]:
films = spark.read.format("jdbc").options(
    url=jdbc_url,
    query="SELECT * FROM public.film WHERE rental_duration > 3",
    **connection_properties
).load()

display(films)

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
films_grouped = films.groupBy("rental_duration").count().orderBy("count", ascending=False)
display(films_grouped)

Databricks visualization. Run in Databricks to view.

## Deleting all schemas in untiy catalog

In [0]:
spark.sql("DROP CATALOG aws CASCADE")

In [0]:
%sql
drop TABLE workspace.default.test_table;

## Unity Catalog Aurora
### Connecting using connect: Lakehouse federation
<font color='red'>Bear in mind Aurora Postgresql instance is no longer avaiable</font>

1. Defining connection (ex. SQL) with `Databrics secrets`

In [0]:
%sql
CREATE CONNECTION conn_aurora TYPE postgresql
OPTIONS (
  host secret('dvdrental', 'aws_aurora_pg_endpoint'),
  port '5432',
  user secret('dvdrental','db_user'),
  password secret('dvdrental','db_password')
);

2. Creating Foreign Catalog

In [0]:
%sql
CREATE FOREIGN CATALOG aurora_db
  USING CONNECTION conn_aurora
  OPTIONS (database 'dvdrental');

In [0]:
display(_sqldf)

3. Queries: like SELECT * FROM aurora_db.public.my_table. This will fetch records from Aurora and synchronize meta-data (table names and column names) on the fly.

In [0]:
%sql
SELECT f.title AS film_title,
       ai.first_name,
       ai.last_name
FROM aurora_db.public.film_actor fa
JOIN aurora_db.public.actor_info ai ON fa.actor_id = ai.actor_id
JOIN aurora_db.public.film f ON fa.film_id = f.film_id
WHERE fa.actor_id IN (1, 5)
ORDER BY ai.last_name ASC, ai.first_name ASC;

## Saving data as databricks table

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA DEFAULT")

from pyspark.sql.functions import col, min as spark_min, max as spark_max
from pyspark.ml.feature import Bucketizer

num_chunks = 4

# Collect rental durations using DataFrame operations
rental_durations = films.select("rental_duration").agg(
    spark_min("rental_duration").alias("min_duration"),
    spark_max("rental_duration").alias("max_duration")
).first()

min_duration = rental_durations.min_duration
max_duration = rental_durations.max_duration
splits = [min_duration] + [
    min_duration + i * (max_duration - min_duration) / num_chunks for i in range(1, num_chunks)
] + [max_duration + 1]

bucketizer = Bucketizer(splits=splits, inputCol="rental_duration", outputCol="duration_bin")
films_binned = bucketizer.transform(films)

for i in range(num_chunks):
    chunk = films_binned.filter(col("duration_bin") == float(i))
    chunk.write.format("delta").mode("append").saveAsTable(f"films_rental_bin_{i}")

In [0]:
%sql
use catalog `workspace`; select * from `default`.`films_rental_bin_0` limit 100;

In [0]:
__dw_df_0_in = df_pg
__dw_df_0 = __dw_df_0_in.filter("(rental_duration >= 5)")
__dw_df_0 = __dw_df_0.filter("length > 10").sort('title', ascending=True)
__dw_df_0.createOrReplaceTempView('__dw_df_0')
display(__dw_df_0)

## Interacting with AWS Bedrock

In [0]:
from openai import OpenAI

REGION = "eu-west-1"
MODEL_ID = "openai.gpt-oss-120b"

def main() -> None:
    api_key = dbutils.secrets.get(
        scope="aws",
        key="bearer_token_bedrock"
    )
    if not api_key:
        raise RuntimeError(
            "Missing API key. Set OPENAI_API_KEY or AWS_BEARER_TOKEN_BEDROCK "
            "before running this script."
        )

    client = OpenAI(
        api_key=api_key,
        base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    )

    try:
        response = client.responses.create(
            model=MODEL_ID,
            input=[
                {
                    "role": "user",
                    "content": "Write a one-sentence bedtime story about a unicorn.",
                }
            ],
        )
        print(response.output_text)
    except Exception as e:
        print(f"API call failed: {str(e)}")

if __name__ == "__main__":
    main()

In [0]:
import requests
print(requests.get("https://google.com").status_code)

In [0]:
from openai import OpenAI

REGION = "eu-west-1"
MODEL_ID = "openai.gpt-oss-120b"
API_KEY = dbutils.secrets.get(
    scope="aws",
    key="bearer_token_bedrock"
)

def call_bedrock_openai(prompt: str) -> str:
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1"
    )
    response = client.responses.create(
        model=MODEL_ID,
        input=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.output_text
    )
    return response.output_text

In [0]:
call_bedrock_openai("Explain Databricks in one sentence")

In [0]:
from openai import OpenAI

try:
    API_KEY = dbutils.secrets.get(
        "aws",
        "bearer-token-bedrock"
    )
except Exception as e:
    raise RuntimeError(
        "Failed to retrieve secret. Please verify the scope and key."
    ) from e

REGION = "eu-west-1"
MODEL_ID = "openai.gpt-oss-120b"

def get_bedrock_client(api_key: str) -> OpenAI:
    client = OpenAI(
        api_key=api_key,
        base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1"
    )
    return client

def call_bedrock_openai(prompt: str, client: OpenAI) -> str:
    response = client.responses.create(
        model=MODEL_ID,
        input=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.output_text

In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
def llm_udf(prompt: str) -> str:

    try:
        return call_bedrock_openai(prompt)
    except Exception as e:
        return f"error: {str(e)}"

spark_udf = udf(llm_udf, StringType())

In [0]:
df = spark.createDataFrame([
    ("Explain Databricks in one sentence",),
    ("Write a poem about cloud computing",),
    ("What is AWS Bedrock?",)
], ["prompt"])

df = df.withColumn("response", spark_udf("prompt"))

display(df)

In [0]:
from pyspark.sql.functions import udf, lit
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def greet(name: str, greeting: str = "Hello") -> str:
    if name is None:
        return "No name provided"
    return f"{greeting}, {name}!"

# Use the UDF in a DataFrame
df = df.withColumn("greeting_message", greet(df["name"], lit("Hi")))
display(df)